In [1]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import scanpy as sc
import squidpy as sq
import enrichmap as em
import anndata
import pandas as pd
import numpy as np
import gseapy as gp
import decoupler as dc

from matplotlib.colors import ListedColormap
from scipy.sparse import csr_matrix
from scipy.spatial.distance import cdist

In [4]:
sc.set_figure_params(frameon=False)
sc.settings._vector_friendly = False

In [5]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
working_dir = project_dir + ""
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = working_dir + "processed_data/"
os.makedirs(processed_data, exist_ok=True)


In [6]:
# set random seed
np.random.seed(0)

In [7]:
from scipy.spatial import KDTree

def simulate_visium(
    seed=0,
    n_genes=3000,
    n_signature_genes=20,
    n_clusters=4,
    target_cluster=1,
    n_spots_range=(1500, 4000),
    spot_spacing_um=100,
    spot_diameter_um=55,
    mean_counts_per_spot=5000,
    dispersion_range=(0.1, 2.0),
    fold_change_range=(1.5, 5.0),
    dropout_rate=0.1,
    n_patches_per_cluster=100,
    smoothing_iterations=1,
):
    """
    Simulate a 10x Visium spatial transcriptomics dataset.

    The grid layout matches the Visium platform: a square capture area
    with hexagonally arranged spots at 100 um centre-to-centre spacing.
    The number of spots is drawn randomly from ``n_spots_range`` to
    mimic variable tissue coverage across real slides. This is achieved
    by varying the capture area dimensions per simulation.

    Clusters are formed by seeding ``n_patches_per_cluster`` random foci
    per cluster and then iteratively smoothing assignments by majority
    vote within a local neighbourhood, producing irregular, tissue-like
    patches instead of clean Voronoi regions.

    Returns (adata, sig_genes, fold_changes, target_label).
    """
    rng = np.random.default_rng(seed)

    # Variable spot count via variable capture area
    # Draw a target spot count, then compute the grid side that
    # produces approximately that many spots in a hexagonal grid.
    # For a hex grid: n_spots ≈ n_rows × n_cols ≈ side^2 / 0.866
    n_spots_target = rng.integers(
        n_spots_range[0],
        n_spots_range[1] + 1,
    )
    grid_side = int(np.ceil(np.sqrt(n_spots_target * 0.866)))
    capture_area_um = grid_side * spot_spacing_um

    # Hexagonal grid within square capture area
    n_cols = int(capture_area_um / spot_spacing_um)
    n_rows = int(capture_area_um / (spot_spacing_um * np.sqrt(3) / 2))

    x_coords, y_coords = [], []
    for row in range(n_rows):
        for col in range(n_cols):
            x = col * spot_spacing_um + (row % 2) * (spot_spacing_um / 2)
            y = row * spot_spacing_um * np.sqrt(3) / 2
            if x <= capture_area_um and y <= capture_area_um:
                x_coords.append(x)
                y_coords.append(y)

    x_coords = np.array(x_coords)
    y_coords = np.array(y_coords)
    n_spots = len(x_coords)
    coords = np.column_stack([x_coords, y_coords])

    # Patchy clusters: multiple seed foci per cluster + spatial smoothing
    # Each cluster gets n_patches_per_cluster random seed points so the
    # same label can appear as several disconnected blobs across the tissue,
    # mimicking infiltrating immune zones or tumour patches.
    margin = capture_area_um * 0.15
    total_seeds = n_clusters * n_patches_per_cluster
    seed_cluster_ids = np.repeat(np.arange(n_clusters), n_patches_per_cluster)

    seed_coords = np.column_stack([
        rng.uniform(margin, capture_area_um - margin, total_seeds),
        rng.uniform(margin, capture_area_um - margin, total_seeds),
    ])

    dists = cdist(coords, seed_coords)
    nearest_seed = np.argmin(dists, axis=1)
    cluster_ids = seed_cluster_ids[nearest_seed]

    # Iterative majority-vote smoothing: each spot adopts the most common
    # label among its spatial neighbours, creating organic, rounded boundaries.
    tree = KDTree(coords)
    smoothing_radius = spot_spacing_um * 2.5
    neighbors = tree.query_ball_point(coords, smoothing_radius)
    for _ in range(smoothing_iterations):
        cluster_ids = np.array([
            np.bincount(cluster_ids[nbrs], minlength=n_clusters).argmax()
            for nbrs in neighbors
        ])

    cluster_labels = np.array([f"cluster_{i + 1}" for i in cluster_ids])
    target_label = f"cluster_{target_cluster + 1}"

    # Gene names
    gene_names = [f"GENE_{i + 1:05d}" for i in range(n_genes)]

    # Signature genes with graded fold changes
    sig_indices = rng.choice(n_genes, n_signature_genes, replace=False)
    fold_changes = rng.uniform(
        fold_change_range[0],
        fold_change_range[1],
        n_signature_genes,
    )
    order = np.argsort(-fold_changes)
    sig_indices = sig_indices[order]
    fold_changes = fold_changes[order]
    sig_genes = [gene_names[i] for i in sig_indices]

    # Expression matrix (negative binomial)
    gene_means = np.exp(rng.normal(1.5, 1.0, n_genes))
    gene_means = np.clip(gene_means, 0.1, 100)

    dispersions = rng.uniform(
        dispersion_range[0],
        dispersion_range[1],
        n_genes,
    )

    lib_size_factors = np.exp(rng.normal(0, 0.3, n_spots))
    lib_size_factors *= mean_counts_per_spot / (
        gene_means.sum() * lib_size_factors.mean()
    )

    X = np.zeros((n_spots, n_genes), dtype=np.float32)
    target_mask = cluster_labels == target_label

    for g in range(n_genes):
        mu = gene_means[g] * lib_size_factors

        if g in sig_indices:
            fc_idx = np.where(sig_indices == g)[0][0]
            mu[target_mask] *= fold_changes[fc_idx]

        r = dispersions[g]
        p = r / (r + mu)
        counts = rng.negative_binomial(r, p)

        dropout_mask = rng.random(n_spots) < dropout_rate
        counts[dropout_mask] = 0

        X[:, g] = counts

    # Assemble AnnData
    adata = anndata.AnnData(X=csr_matrix(X))
    adata.var_names = gene_names
    adata.obs_names = [f"spot_{i}" for i in range(n_spots)]
    adata.obs["cluster"] = pd.Categorical(cluster_labels)
    adata.obsm["spatial"] = coords

    adata.raw = adata.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    return adata, sig_genes, fold_changes, target_label


In [8]:
adata, sig_genes, fold_changes, target_label = simulate_visium(seed=0)


libgomp: Thread creation failed: Resource temporarily unavailable


: 

In [ ]:
cluster_palettes = {
    "cluster_1": "#F6C141",
    "cluster_2": "#882E72",
    "cluster_3": "#7BAFDE",
    "cluster_4": "#F4A582"
}

# Convert 'cluster' column to categorical dtype
adata.obs["cluster"] = pd.Categorical(adata.obs["cluster"])

# Now you can access the categories
clusters = adata.obs["cluster"].cat.categories

# Reorder the palette list to match the category order
palette_list = [cluster_palettes[c] for c in clusters]

# Create a ListedColormap
cmap = ListedColormap(palette_list)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color="cluster",
    img_alpha=0,
    size=20,
    shape=None,
    palette=cmap,
    save=figure_dir + "simulated_spatial_scatter_clusters.pdf",
)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

### EnrichMap

In [ ]:
em.tl.score(adata, gene_set=sig_genes)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color=["enrichmap_score"],
    size=20,
    vcenter=0,
    cmap="RdBu_r",
    shape=None,
    save="simulated_spatial_scatter_enrichmap_score.pdf",
)

### AUCell

In [ ]:
# convert the signature dict to a long format
signatures_dict = {"cluster_2": sig_genes}

signatures = pd.DataFrame(
    [(key, gene) for key, genes in signatures_dict.items() for gene in genes],
    columns=["source", "target"],
)

dc.mt.aucell(adata, net=signatures)
score_aucell = dc.pp.get_obsm(adata=adata, key="score_aucell")
adata.obs["aucell_score"] = score_aucell.obsm["score_aucell"]

## `score_genes()`/`AddModuleScore()`

In [ ]:
sc.tl.score_genes(adata, sig_genes, score_name="scanpy_score", use_raw=False)

## ssGSEA

In [ ]:
# If adata.X is sparse, convert to dense first
X_dense = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X

# Create DataFrame using var names as columns and obs names as index
gene_expr = pd.DataFrame(X_dense.T, index=adata.var_names, columns=adata.obs_names)

In [ ]:
gene_expr.index = gene_expr.index.str.upper()
signatures_dict = {k: [g.upper() for g in v] for k, v in signatures_dict.items()}

In [ ]:
ssgsea = gp.ssgsea(
    data=gene_expr,
    gene_sets=signatures_dict,
    outdir=None,
    sample_norm_method="rank",
    no_plot=True,
    min_size=2
)

In [ ]:
nes = ssgsea.res2d.pivot(index="Term", columns="Name", values="NES")
adata.obs["ssgsea_score"] = nes.loc["cluster_2"].T.astype(float).reindex(adata.obs_names)

### GSVA

In [ ]:
gsva = gp.gsva(
    data=gene_expr,
    gene_sets=signatures_dict,
    outdir=None,
    min_size=2
)

In [ ]:
es = gsva.res2d.pivot(index="Term", columns="Name", values="ES")
adata.obs["gsva_score"] = es.loc["cluster_2"].T.astype(float).reindex(adata.obs_names)

### Z score

In [ ]:
gene_set = signatures_dict["cluster_2"]
z_score = gene_expr.loc[gene_set].sum(axis=0) / np.sqrt(len(gene_set))
adata.obs["z_score"] = z_score

In [ ]:
methods = ["enrichmap_score", "aucell_score", "z_score", "scanpy_score", "ssgsea_score", "gsva_score"]

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color=methods,
    img_alpha=0,
    shape=None,
    size=20,
    ncols=3,
    cmap="RdBu_r",
    save="simulated_spatial_scatter_score_comparison.pdf",
)

In [ ]:
em.pl.spatial_metrics(
    adata,
    methods,
    metric="Moran's I",
    figsize=(3, 2),
    save="simulated_spatial_metrics_moransI.pdf",
)

In [ ]:
em.pl.spatial_metrics(
    adata,
    methods,
    metric="Geary's C",
    figsize=(3, 2),
    save="simulated_spatial_metrics_gearysC.pdf",
)

In [ ]:
em.pl.variogram(
    adata,
    score_keys=methods,
    save="simulated_variogram_method_comparison.pdf",
)

In [ ]:
em.pl.variogram_all(
    adata,
    score_keys=methods,
    save="simulated_variogram_method_comparison_all.pdf",
)

In [ ]:
em.pl.morans_correlogram(
    adata,
    score_key="enrichmap_score",
    save="simulated_morans_correlogram_enrichmap_score.pdf",
)

In [ ]:
em.pl.gene_contributions_pca(
    adata,
    score_key="enrichmap_score",
    top_n_genes=10,
    save="simulated_gene_contributions_pca_enrichmap_score.pdf",
)

In [ ]:
em.pl.gene_contributions_heatmap(
    adata,
    score_key="enrichmap",
    order_spots="coherence",
    top_n_genes=3,
    bottom_n_genes=2,
    figsize=(3, 3),
    cmap="RdBu_r",
    save="simulated_gene_contributions_heatmap_enrichmap_score.pdf",
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color=[g for g in sig_genes],
    size=20,
    shape=None,
    use_raw=False,
    cmap="PuOr_r",
    save="simulated_spatial_scatter_example_genes.pdf",
)